In [ ]:
# 01 - SETUP: imports, paths e exibição
import os
from pathlib import Path
import math
import numpy as np
import pandas as pd
from datetime import datetime
import plotly.graph_objects as go
import plotly.express as px
from scipy.stats import norm, lognorm, beta
from tqdm import trange

# Diretórios
BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = DATA_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pd.options.display.float_format = "R$ {:,.2f}".format

print("Notebook 4 — Monte Carlo iniciado.")
print("Base:", BASE_DIR)


In [ ]:
# 02 - GLOSSÁRIO (reuso ou fallback)
try:
    from core.glossary import obter_explicacao
    print("Usando core.glossary.obter_explicacao()")
except Exception:
    print("core.glossary não disponível — usando fallback curto.")
    _GLOSS = {
        "MRR": {"layman": "Receita Recorrente Mensal — receita de assinaturas a cada mês."},
        "Caixa": {"layman": "Saldo acumulado, quanto de dinheiro a empresa tem disponível."},
        "Churn": {"layman": "Percentual de clientes que cancelam por mês."},
        "MonteCarlo": {"layman": "Simulações aleatórias para estimar incerteza."},
        "Runway": {"layman": "Quantos meses o caixa atual aguenta com o gasto atual."}
    }
    def obter_explicacao(chave, publico="layman"):
        return _GLOSS.get(chave, {}).get(publico, "")


In [ ]:
# 03 - IMPORTAR GERADOR DE PROJECOES (ou usar fallback)
try:
    from core.engine import gerar_projecao_financeira
    print("Usando gerar_projecao_financeira() do core.engine")
except Exception:
    print("gerar_projecao_financeira() não encontrado. Usando fallback interno (compatível).")
    def gerar_projecao_financeira(p, meses=None):
        # Implementação compatível simplificada (mesma lógica do Notebook1)
        meses = meses or int(p.get("meses_projecao", 36))
        start_date = datetime.fromisoformat(p.get("data_inicio", "2025-11-01"))
        idx = [ (start_date.replace(day=1) + pd.DateOffset(months=i)).strftime("%Y-%m") for i in range(meses) ]
        trafego = np.zeros(meses); trafego[0] = p.get("trafego_inicial", 1000)
        for t in range(1, meses):
            trafego[t] = trafego[t-1] * (1 + p.get("crescimento_trafego_mensal", 0.2))
        trials = np.round(trafego * p.get("visitante_para_trial", 0.05)).astype(int)
        novos_raw = np.round(trials * p.get("trial_para_pago", 0.15)).astype(int)
        ramp = np.ones(meses); ramp[:6] = np.linspace(0.6, 1.0, 6)
        novos = (novos_raw * ramp).astype(int)
        clientes = np.zeros(meses); clientes[0] = p.get("usuarios_pagos_iniciais", 50)
        for t in range(1, meses):
            clientes[t] = clientes[t-1] * (1 - p.get("churn_mensal", 0.04)) + novos[t]
        cust_lite = clientes * p.get("mix_lite",0.5)
        cust_trader = clientes * p.get("mix_trader",0.3)
        cust_pro = clientes * p.get("mix_pro",0.2)
        mrr = cust_lite*p.get("preco_lite",69.9) + cust_trader*p.get("preco_trader",99.0) + cust_pro*p.get("preco_pro",159.0)
        marketing = np.zeros(meses)
        for t in range(meses):
            marketing[t] = p.get("marketing_minimo",1000.0) if t==0 else max(p.get("marketing_minimo",1000.0), mrr[t-1]*p.get("marketing_reinvest_pct",0.15))
        folha = np.zeros(meses)
        folha += p.get("salario_fundador",10000)*(1+p.get("encargos_pct",0.68))
        for t in range(11, meses):
            folha[t] += p.get("salario_eng",8000)*(1+p.get("encargos_pct",0.68))
        for t in range(17, meses):
            folha[t] += p.get("salario_cs",5000)*(1+p.get("encargos_pct",0.68))
        custo_ia = clientes * p.get("custo_ia_por_usuario",5.0)
        taxas = mrr * p.get("taxa_pagamento_pct",0.03)
        cogs = custo_ia + taxas + marketing
        infra = np.full(meses, p.get("infra_base_mensal",709.0))
        opex = folha + infra
        receita = mrr.copy()
        lucro_bruto = receita - (custo_ia + taxas)
        ebitda = lucro_bruto - (folha + infra)
        depreciacao = np.full(meses, p.get("capex_unico",8000.0)/60.0)
        impostos = np.where(ebitda>0, ebitda * p.get("taxa_impostos_pct",0.06), 0.0)
        lucro_liquido = ebitda - depreciacao - impostos
        caixa = np.zeros(meses)
        caixa[0] = p.get("caixa_inicial",50000.0) + lucro_liquido[0] - p.get("capex_unico",8000.0)
        for t in range(1, meses):
            caixa[t] = caixa[t-1] + lucro_liquido[t]
        arpu = np.where(clientes>0, mrr / clientes, 0.0)
        ltv = np.where(p.get("churn_mensal",0.04)>0, arpu / p.get("churn_mensal",0.04), 0.0)
        cac = np.full(meses, np.nan)
        for t in range(meses):
            s = max(0, t-11)
            new12 = novos[s:t+1].sum()
            mar12 = marketing[s:t+1].sum()
            cac[t] = (mar12 / new12) if new12>0 else np.nan
        df = pd.DataFrame({ "mes": idx, "trafego": trafego.astype(int), "trials": trials, "novos_pagantes": novos,
                           "clientes_ativos": np.round(clientes).astype(int), "mrr": mrr.round(2),
                           "marketing": marketing.round(2), "custo_ia": custo_ia.round(2), "taxas_pag": taxas.round(2),
                           "cogs": cogs.round(2), "folha": folha.round(2), "infra": infra.round(2), "opex_total": opex.round(2),
                           "lucro_bruto": lucro_bruto.round(2), "ebitda": ebitda.round(2), "depreciacao": depreciacao.round(2),
                           "impostos": impostos.round(2), "lucro_liquido": lucro_liquido.round(2), "caixa": caixa.round(2),
                           "arpu": np.round(arpu,2), "ltv": np.round(ltv,2), "cac": np.round(cac,2) } )
        return df


In [ ]:
# 04 - PREMISSAS BASE (edite para calibrar as simulações)
premissa_base = {
    "data_inicio":"2025-11-01",
    "meses_projecao":36,
    "trafego_inicial":1000,
    "crescimento_trafego_mensal":0.20,
    "visitante_para_trial":0.05,
    "trial_para_pago":0.15,
    "usuarios_pagos_iniciais":50,
    "churn_mensal":0.04,
    "preco_lite":69.9,
    "preco_trader":99.0,
    "preco_pro":159.0,
    "mix_lite":0.5,
    "mix_trader":0.3,
    "mix_pro":0.2,
    "custo_ia_por_usuario":5.0,
    "taxa_pagamento_pct":0.03,
    "infra_base_mensal":709.0,
    "marketing_minimo":1000.0,
    "marketing_reinvest_pct":0.15,
    "salario_fundador":10000.0,
    "salario_eng":8000.0,
    "salario_cs":5000.0,
    "encargos_pct":0.68,
    "capex_unico":8000.0,
    "taxa_impostos_pct":0.06,
    "caixa_inicial":50000.0
}


In [ ]:
# 05 - BASELINE DETERMINÍSTICA
df_base = gerar_projecao_financeira(premissa_base, meses=premissa_base["meses_projecao"])
# converter mes para datetime para gráficos
try:
    df_base["mes_dt"] = pd.to_datetime(df_base["mes"] + "-01")
except Exception:
    df_base["mes_dt"] = pd.date_range("2025-11-01", periods=len(df_base), freq="MS")
    
# mostrar resumo rápido (MRR, caixa final, menor caixa, break-even)
mr1 = df_base.loc[0,"mrr"]
mr12 = df_base.loc[min(11,len(df_base)-1),"mrr"]
mr36 = df_base.loc[len(df_base)-1,"mrr"]
menor_idx = df_base["caixa"].idxmin(); menor_mes = df_base.loc[menor_idx,"mes"]; menor_val = df_base.loc[menor_idx,"caixa"]
break_idx = df_base.index[df_base["ebitda"]>0]; break_mes = df_base.loc[break_idx[0],"mes"] if len(break_idx)>0 else None

print(f"MRR M1: R$ {mr1:,.2f}  |  MRR M12: R$ {mr12:,.2f}  |  MRR M36: R$ {mr36:,.2f}")
print(f"Menor caixa: {menor_mes} => R$ {menor_val:,.2f}")
print(f"Break-even (primeiro EBITDA>0): {break_mes}")


In [ ]:
# 06 - MONTE CARLO: definir distribuições e executar simulações
N = 5000  # número de simulações
meses = premissa_base["meses_projecao"]

# Distribuições (paramétricas) - ajuste conforme conhecimento
#  - crescimento_trafego_mensal: lognormal around base (so growth factor >0)
#  - churn_mensal: normal truncated
#  - custo_ia_por_usuario: normal truncated
rng = np.random.default_rng(42)

# Parameters for sampling (calibrados na base)
growth_mu = math.log(1 + premissa_base["crescimento_trafego_mensal"])   # for lognormal
growth_sigma = 0.25  # higher sigma => more uncertainty in growth

churn_mu = premissa_base["churn_mensal"]
churn_sigma = 0.01

custo_ia_mu = premissa_base["custo_ia_por_usuario"]
custo_ia_sigma = 1.0

# pré-alocar resultados
final_cash = np.zeros(N)
cash_paths = np.zeros((N, meses))

# loop (com tqdm)
for i in trange(N, desc="Running Monte Carlo sims"):
    # sample parameters
    # growth sampled as multiplicative factor per month (we will convert to monthly rate)
    sampled_growth_factor = rng.lognormal(mean=growth_mu, sigma=growth_sigma)  # >0
    sampled_growth_rate = sampled_growth_factor - 1.0
    sampled_churn = np.clip(rng.normal(churn_mu, churn_sigma), 0.005, 0.2)
    sampled_custo_ia = np.clip(rng.normal(custo_ia_mu, custo_ia_sigma), 0.5, 50.0)
    
    # build local premissa
    p = premissa_base.copy()
    p["crescimento_trafego_mensal"] = float(sampled_growth_rate)
    p["churn_mensal"] = float(sampled_churn)
    p["custo_ia_por_usuario"] = float(sampled_custo_ia)
    
    # run deterministic engine
    df_sim = gerar_projecao_financeira(p, meses=meses)
    final_cash[i] = df_sim.loc[len(df_sim)-1, "caixa"]
    # keep full path for percentiles
    cash_paths[i, :] = df_sim["caixa"].values


In [ ]:
# 07 - Estatísticas de saída
percentis = np.percentile(final_cash, [1,5,10,25,50,75,90,95,99])
p_neg = (final_cash < 0).mean()  # prob. de caixa negativo no final
mediana = np.median(final_cash)
media = np.mean(final_cash)
std = np.std(final_cash)

print("Resumo Monte Carlo (Caixa final, Mês 36)")
print(f"Simulações: {N}")
print(f"Média: R$ {media:,.2f}  |  Mediana: R$ {mediana:,.2f}  |  Std: R$ {std:,.2f}")
print("Percentis (1,5,10,25,50,75,90,95,99):", ["R$ {:,.0f}".format(p) for p in percentis])
print(f"Probabilidade Caixa final < 0: {p_neg*100:.2f}%")


In [ ]:
# 08 - HISTOGRAMA DO CAIXA FINAL
fig = px.histogram(final_cash, nbins=100, title="Distribuição do Caixa Final (Mês 36) — Monte Carlo",
                   labels={"value":"Caixa final (R$)"})
fig.update_layout(xaxis_title="Caixa final (R$)", yaxis_title="Frequência")
# anota percentis
for q in [5,25,50,75,95]:
    xq = np.percentile(final_cash, q)
    fig.add_vline(x=xq, line_dash="dash", annotation_text=f"P{q} = R$ {xq:,.0f}", annotation_position="top left")
fig.add_annotation(text=obter_explicacao("MonteCarlo","layman"),
                   xref="paper", yref="paper", x=0, y=-0.25, showarrow=False, align="left")
fig.show()


In [ ]:
# 09 - BANDAS PERCENTIS AO LONGO DO TEMPO (10-90, 25-75, mediana)
percentis_time = {}
qs = [5,25,50,75,95]
for q in qs:
    percentis_time[q] = np.percentile(cash_paths, q, axis=0)

x = df_base["mes_dt"]

fig = go.Figure()
# 10-90 area (use 5 and 95 to approximate)
fig.add_trace(go.Scatter(x=x, y=percentis_time[95], fill=None, mode='lines', line=dict(color='rgba(0,0,0,0)'),
                         showlegend=False))
fig.add_trace(go.Scatter(x=x, y=percentis_time[5], fill='tonexty', mode='lines', line=dict(color='rgba(0,0,0,0)'),
                         fillcolor='rgba(200,200,255,0.2)', name='P5-P95'))
# 25-75
fig.add_trace(go.Scatter(x=x, y=percentis_time[75], fill=None, mode='lines', line=dict(color='rgba(0,0,0,0)'),
                         showlegend=False))
fig.add_trace(go.Scatter(x=x, y=percentis_time[25], fill='tonexty', mode='lines', line=dict(color='rgba(0,0,0,0)'),
                         fillcolor='rgba(100,150,255,0.3)', name='P25-P75'))
# median
fig.add_trace(go.Scatter(x=x, y=percentis_time[50], mode='lines', name='Mediana (P50)', line=dict(color='blue', width=3)))

fig.update_layout(title="Faixa de Incerteza do Saldo de Caixa ao Longo do Tempo (Monte Carlo)",
                  xaxis_title="Mês", yaxis_title="Saldo de Caixa (R$)",
                  hovermode="x unified")
fig.add_annotation(text=("Legenda: áreas mostram intervalos de confiança das simulações. "
                         "Ex.: faixa P25-P75 contém o 50% central das simulações."),
                   xref="paper", yref="paper", x=0, y=-0.25, showarrow=False, align="left")
fig.show()


In [ ]:
# 10 - Probabilidade de caixa negativo por mês
prob_negative = (cash_paths < 0).mean(axis=0)  # proporção de simulações com caixa<0 por mês

fig = go.Figure()
fig.add_trace(go.Scatter(x=x, y=prob_negative, mode='lines+markers', line=dict(color='red'), name='P(Caixa < 0)'))
fig.update_layout(title="Probabilidade de Caixa Negativo por Mês (Monte Carlo)",
                  xaxis_title="Mês", yaxis_title="Probabilidade (0-1)")
fig.add_annotation(text=("Interpretação: valor 0.2 = 20% das simulações apresentaram caixa negativo naquele mês."),
                   xref="paper", yref="paper", x=0, y=-0.25, showarrow=False, align="left")
fig.show()


In [ ]:
# 11 - Conclusões textuais (prontas para capturar em relatório)
print("✅ Conclusões e recomendações (automáticas):\n")
print(f"- Probabilidade de caixa final negativo (M36): {p_neg*100:.2f}%")
print(f"- Mediana do caixa final: R$ {mediana:,.2f}")
print(f"- Percentil 5 do caixa final: R$ {percentis[1]:,.0f}  (cenário conservador)")
print()
if p_neg > 0.5:
    print("🚨 RISCO ALTO: Mais da metade das simulações terminam com caixa negativo. Necessário aporte ou reduzir burn imediatamente.")
elif p_neg > 0.2:
    print("⚠️ RISCO MODERADO: entre 20% e 50% das simulações falham. Recomendado testar redução de OPEX e plano de captação.")
else:
    print("✅ RISCO BAIXO: menos de 20% das simulações terminam em falha — cenário relativamente seguro.")
print()
print("Recomendações práticas:")
print("- Simular cenários de aporte (seed) e reexecutar Monte Carlo para medir impacto na probabilidade de sobrevivência.")
print("- Priorizar medidas que aumentem a mediana do caixa (aumentar crescimento, reduzir churn, cortar OPEX) — use o tornado para identificar melhor alavanca.")
print("- Incorporar correlações entre variáveis (ex.: crescimento e churn) nas próximas iterações para maior realismo.")


In [ ]:
# 12 - EXPORT (opcional) — salvar arrays resumidos e percentis para uso no app/relatório
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
np.save(OUTPUT_DIR / f"mc_final_cash_{ts}.npy", final_cash)
pd.DataFrame(percentis_time).to_csv(OUTPUT_DIR / f"mc_percentiles_time_{ts}.csv", index=True)
pd.DataFrame({ "final_cash": final_cash }).to_csv(OUTPUT_DIR / f"mc_final_cash_{ts}.csv", index=False)

print("Arquivos de resumo salvos em:", OUTPUT_DIR)
